# Step 2: Build your bilayer structure
In this section, we change the input peptide alignment, multiply the 2 x 2 building block 5 times, and visualise the output bilayer structure. We select energy minimisation to refine the overall assembled structure.

## Edit your parameters here

In [1]:
Root_in = 'data_b_strands/'
Input_filename = 'F8.pdb' #@param {type:"string"}
Num_residues_per_peptide = 8 #@param {type:"integer"}
Reference_segment = '2-8' #@param {type:"string"}
Alignment_pep_orientations = 'aaa' #@param ['aaa','apa','aap','app','paa','ppa','pap','ppp']
Output_filename = 'F8_bilayer.pdb' #@param {type:"string"}
Start_res_at_inner_face = False #@param {type:"boolean"}
Alignment_face_to_face = True #@param {type:"boolean"}
Energy_minimisation = True #@param {type:"boolean"}
Show_the_structure = True #@param {type:"boolean"}

## Run this code to build your bilayer structure

In [2]:
root = 'data_bilayers/'

from fibrilgen_local.pep2unit import *
from fibrilgen_local.reordering import *
import os

def mol2pdb_convertsion(root,filename):
	os.system(f'obabel {root}{filename}.mol -O {root}{filename}_temp.pdb')
	pymol.cmd.load(f'{root}{filename}_temp.pdb')
	pymol.cmd.save(f'{root}{filename}.pdb',f'{filename}_temp')
	os.system(f'rm {root}{filename}_temp.pdb')

## Reset
pymol.cmd.delete('all')
pymol.cmd.reset()

## Load PDB
pymol.cmd.load(Root_in+Input_filename)

## Create a periodic unit
name = Input_filename[:-4]
resi_start, resi_end = Reference_segment.split('-')
# INPUT: (peptide, start residue index, end residue index)
unit = create_pep_unit(name,int(resi_start),int(resi_end),Start_res_at_inner_face)

## --- Examples of sheet structures ---
# INPUT: (peptide unit)
sheet = create_sheet(unit)
# INPUT: (peptide alignment aaa/apa/aap/app/paa/ppa/pap/ppp, does two beta-sheet aligned face-to-face, num of units per sheet)
sheet.build_a_plain_sheet(Alignment_pep_orientations,Alignment_face_to_face,5)
## --- End ---

## Save the structure
pymol.cmd.save(root+'bilayer.pdb','plain_sheet')
ordering_indices(root,'bilayer.pdb','bilayer_indexed.pdb')

## Import openbabel and optimization library
if Energy_minimisation:
  from fibrilgen_local.optimize import *
  # Remove steric clashes
  pymol.cmd.create('plain_sheet_em','plain_sheet')
  minimize('plain_sheet_em',nsteps0=10)
  # Save the structure
  pymol.cmd.save(root+'bilayer_em.mol','plain_sheet_em')
  mol2pdb_convertsion(root,'bilayer_em')
  # Index the structure (openbabel energy minimisation changes peptide indices)
  ordering_indices(root,'bilayer_em.pdb','bilayer_em_indexed.pdb')
  reordering_indices(root,'bilayer_indexed','bilayer_em_indexed','bilayer_em_indexed2',20,Num_residues_per_peptide)
  # Calculate the energy
  os.system(f'obabel {root}bilayer_em_indexed2.pdb -otxt --energy --append "Energy" --ff MMFF94')
  # Ouput the structure for visualisation
  os.system(f'cp {root}bilayer_em_indexed2.pdb {root}{Output_filename}')
else:
  # Ouput the structure for visualisation
  os.system(f'cp {root}bilayer.pdb {root}{Output_filename}')

## Show the structure
if Show_the_structure:
  import py3Dmol
  with open(root+Output_filename) as ifile:
      system = "".join([x for x in ifile])
  view = py3Dmol.view(width=900, height=300)
  view.addModelsAsFrames(system)
  view.setStyle({"stick": {'color': 'spectrum'}})
  view.zoomTo()
  view.show()


1 molecule converted


data_bilayers/bilayer_em_indexed2.pdb 5664.08


1 molecule converted


3Dmol.js failed to load for some reason. Please check your browser console for error messages.